# Beautiful Soup 1 — Read a Yahoo Finance quote page

**Goal:** use the supplied Yahoo Finance Apple (`AAPL`) page to learn Beautiful Soup one small operation at a time, then make a compact quote table.

## 1. Fetch the page

First import the two tools used to request and parse HTML.

In [128]:
import requests
from bs4 import BeautifulSoup

In [129]:
symbol = "AAPL"
url = f"https://finance.yahoo.com/quote/{symbol}/"

### `requests.get` or `session.get`?

| | `requests.get(url, headers=headers)` | `session.get(url)` |
|---|---|---|
| headers | passed **every** time | set **once** |
| cookies | forgotten after each call | **kept** for the next calls |
| connection | a new one each time | **reused** (faster) |

One request: both work. Several requests to the same site: use a **session**.

In [130]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/157.0.0.0 Safari/537.36",   # a real Chrome User-Agent
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(headers)   # set ONCE: sent with every request of this session

In [ ]:
response = session.get(url, timeout=20)   # no headers= needed: the session has them

In [132]:
response.raise_for_status()
print("HTTP status:", response.status_code)

HTTP status: 200


In [133]:
soup = BeautifulSoup(response.text, "html.parser")
print("Downloaded characters:", len(response.text))

Downloaded characters: 1230905


### Reuse the session for a second page

Same session: same User-Agent (no headers= to repeat), same open connection, and the cookies if the site sets some (Yahoo sets none here: 0).

In [ ]:
import time

time.sleep(2)   # be polite: a pause between two pages
other = session.get("https://finance.yahoo.com/quote/MSFT/", timeout=20)

print("HTTP status:", other.status_code)
print("User-Agent sent:", other.request.headers["User-Agent"])
print("Cookies kept by the session:", len(session.cookies))

## 2. Inspect one tag

A **tag** is one HTML element. Start with the document title and look at its HTML.

In [134]:
title_tag = soup.title
title_tag

<title>Apple Inc. (AAPL) Stock Price, News, Quote &amp; History - Yahoo Finance</title>

## 3. Use `soup.find`

`find` returns the first matching tag, or `None` when it cannot find one.

In [135]:
first_heading = soup.find("h1")
first_heading

<h1 class="heading yf-ndxd9a">Apple Inc. (AAPL)</h1>

## 4. Use `soup.find_all`

`find_all` returns a list-like `ResultSet` of every matching tag.

In [136]:
heading_tags = soup.find_all("h1")
len(heading_tags)

1

## 5. Find `<span>` tags

The same method can target a common inline tag such as `<span>`. Large pages may contain many spans.

In [137]:
span_tags = soup.find_all("span")
print("Number of <span> tags:", len(span_tags))

Number of <span> tags: 521


In [138]:
example_span = span_tags[0] if span_tags else None
example_span

<span class="language yf-1357fio">English</span>

## 6. Read tag properties

Tags expose a name, text, all attributes, and a `class` attribute when present. Guard against `None` because a page can change.

In [139]:
if example_span is None:
    print("No <span> tag was returned.")
else:
    print("name :", example_span.name)
    print("text :", example_span.get_text(" ", strip=True))
    print("attrs :", example_span.attrs)
    print("class:", example_span.get("class"))

name : span
text : English
attrs : {'class': ['language', 'yf-1357fio']}
class: ['language', 'yf-1357fio']


## 6a. Compare `contents` and text

`contents` keeps a tag's immediate child nodes; `get_text` combines the readable text.

In [140]:
(title_tag.contents, title_tag.get_text(" ", strip=True)) if title_tag else ([], None)

(['Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance'],
 'Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance')

## 6b. Move to a parent tag

The parsed page is a tree. A tag's `parent` is the element that contains it.

In [141]:
first_heading.parent.name if first_heading and first_heading.parent else None

'section'

## 6c. Extract a link safely

Find one anchor tag first. Its `href` may be absent, so use `.get` rather than attribute indexing.

In [142]:
first_link = soup.find("a")
first_link

<a class="secondary-btn-link fin-size-small skip-link yf-119g04z" data-yga='{"yLinkElement":"anchor-tag","yLinkElementType":"link","yTrafficOrigin":"skip-to-nav default","yModuleName":"skip-to-nav","yLinkText":"Skip to navigation"}' data-ylk="elm:anchor-tag;elmt:link;itc:1;source:skip-to-nav%20default;sec:skip-to-nav;slk:Skip%20to%20navigation" href="#navigation-container">Skip to navigation  </a>

## 6d. Read the link URL

This returns the URL or `None` instead of raising an error when there is no link or no `href`.

In [143]:
first_link.get("href") if first_link else None

'#navigation-container'

## 6e. Match an attribute with `find`

Pass an `attrs` dictionary when a meaningful HTML attribute identifies the tag.

### Why the earlier price, change, and percent-change lookups returned `None`

The exact observed reason is a **selector mismatch**. The earlier code searched for legacy `<fin-streamer>` tags with `data-field` values `regularMarketPrice`, `regularMarketChange`, and `regularMarketChangePercent`. In the quote header shown in the browser DevTools screenshot, the values are instead in `<span>` tags with `data-testid` values `qsp-price`, `qsp-price-change`, and `qsp-price-change-percent`. Because those are different tags and attributes, the legacy lookups return `None`.

Browser DevTools **Elements** shows the browser's live DOM. `response.text` is the raw response that `requests` downloaded and that Beautiful Soup parses. If a new selector still returns `None`, inspect the exact HTML in `response.text` before drawing a conclusion about why it is absent.

### Inspect the raw response

Check whether the current selector's `data-testid` is present in the exact HTML that Beautiful Soup receives.

In [144]:
target_testids = [
    "qsp-price",
    "qsp-price-change",
    "qsp-price-change-percent",
]
for testid in target_testids:
    print(testid, f'data-testid=\"{testid}\"' in response.text)

qsp-price True
qsp-price-change True
qsp-price-change-percent True


### Find the price span

In [145]:
price_tag = soup.find(
    "span",
    attrs={"data-testid": "qsp-price"}
)
print(price_tag)

<span class="price yf-1n64cj base" data-testid="qsp-price">344.25 </span>


### Find the change span

In [146]:
change_tag = soup.find(
    "span",
    attrs={"data-testid": "qsp-price-change"}
)
print(change_tag)

<span class="priceChange txt-positive yf-1n64cj base" data-testid="qsp-price-change">+0.08 </span>


### Find the percent-change span

In [147]:
percent_change_tag = soup.find(
    "span",
    attrs={"data-testid": "qsp-price-change-percent"}
)
print(percent_change_tag)

<span class="priceChangePercent txt-positive yf-1n64cj base" data-testid="qsp-price-change-percent">(+0.02%) </span>


## 6f. Match classes with `class_`

Because `class` is a Python keyword, Beautiful Soup uses `class_` in `find` and `find_all`.

In [148]:
soup.find_all("a", class_=first_link.get("class")) if first_link and first_link.get("class") else []

[<a class="secondary-btn-link fin-size-small skip-link yf-119g04z" data-yga='{"yLinkElement":"anchor-tag","yLinkElementType":"link","yTrafficOrigin":"skip-to-nav default","yModuleName":"skip-to-nav","yLinkText":"Skip to navigation"}' data-ylk="elm:anchor-tag;elmt:link;itc:1;source:skip-to-nav%20default;sec:skip-to-nav;slk:Skip%20to%20navigation" href="#navigation-container">Skip to navigation  </a>,
 <a class="secondary-btn-link fin-size-small skip-link yf-119g04z" data-testid="skip-nav-link" data-yga='{"yLinkElement":"anchor-tag","yLinkElementType":"link","yTrafficOrigin":"skip-nav default","yModuleName":"skip-nav","yLinkText":"Skip to main content"}' data-ylk="elm:anchor-tag;elmt:link;itc:1;source:skip-nav%20default;sec:skip-nav;slk:Skip%20to%20main%20content" href="#nimbus-app">Skip to main content  </a>,
 <a class="secondary-btn-link fin-size-small skip-link yf-119g04z" data-yga='{"yLinkElement":"anchor-tag","yLinkElementType":"link","yTrafficOrigin":"skip-to-dock default","yModule

## 6g. Filter by an exact class

A tag's classes are stored as a list. Equality checks the exact class list while filtering an existing `find_all` result.

In [149]:
exact_class_spans = [tag for tag in span_tags if example_span and tag.get("class") == example_span.get("class")]
len(exact_class_spans)

1

## 6h. Use CSS selectors

`select_one` is like `find`, and `select` is like `find_all`, but both use CSS-selector syntax.

In [150]:
soup.select('span[data-testid="qsp-price"]')

[<span class="price yf-1n64cj base" data-testid="qsp-price">344.25 </span>]

## 7. Filter an existing `find_all` result to an exact tag

First reuse the `<span>` tags collected earlier. Then filter that existing result by the meaningful `data-testid` attribute to find the quote price span. This is often clearer than relying on unstable CSS classes.

In [151]:
quote_spans = soup.find_all("span")
print("Number of <span> tags:", len(quote_spans))

Number of <span> tags: 521


In [152]:
price_tag = next(
    (tag for tag in quote_spans if tag.get("data-testid") == "qsp-price"),
    None,
)
price_tag

<span class="price yf-1n64cj base" data-testid="qsp-price">344.25 </span>

In [153]:
regular_price = price_tag.get_text(strip=True) if price_tag else None
print(f"{symbol} regular market price: {regular_price}")

AAPL regular market price: 344.25


## 8. Introduce reusable logic

Put the repeated safe text conversion in a small helper.

In [154]:
def text_or_none(tag):
    return tag.get_text(" ", strip=True) if tag else None

## 9. Introduce CSS selectors

`select_one` finds the first CSS-selector match. The following selector finds the same price span using its `data-testid` attribute.

In [155]:
price_tag_css = soup.select_one('span[data-testid="qsp-price"]')
text_or_none(price_tag_css)

'344.25'

## 10. Build a reusable quote summary

Each lookup remains defensive: a changed page produces `None`, not an attribute error.

In [156]:
def yahoo_quote_summary(parsed_soup, symbol):
    heading = parsed_soup.select_one('[data-testid="quote-header"] h1') or parsed_soup.find("h1")
    price = parsed_soup.select_one('span[data-testid="qsp-price"]')
    change = parsed_soup.select_one('span[data-testid="qsp-price-change"]')
    percent_change = parsed_soup.select_one('span[data-testid="qsp-price-change-percent"]')
    return {
        "symbol": symbol,
        "company": text_or_none(heading),
        "price": text_or_none(price),
        "change": text_or_none(change),
        "change_percent": text_or_none(percent_change),
    }

In [157]:
summary = yahoo_quote_summary(soup, symbol)
summary

{'symbol': 'AAPL',
 'company': 'Apple Inc. (AAPL)',
 'price': '344.25',
 'change': '+0.08',
 'change_percent': '(+0.02%)'}

## 11. Put the extracted record in a table

pandas can represent one extracted record as a one-row table.

In [158]:
import pandas as pd

In [159]:
quote_table = pd.DataFrame([summary])
quote_table

,symbol,company,price,change,change_percent
0,AAPL,Apple Inc. (AAPL),344.25,+0.08,(+0.02%)


## 12. Extract the Yahoo Finance quote-statistics grid

The quote statistics beneath the header are a visually arranged **grid/list of `<li>` elements**, not an HTML table. Typical rows include Previous Close, Open, Bid, Ask, Day's Range, 52 Week Range, Volume, Avg. Volume, Market Cap (intraday), Beta (5Y Monthly), PE Ratio (TTM), EPS (TTM), Earnings Date (est.), Forward Dividend & Yield, and 1y Target Est. Morningstar Rating is optional.

> **Check the downloaded HTML first.** This example can extract statistics only when their matching rows are in `response.text`. DevTools Elements can show a browser-live DOM that differs from the response downloaded by `requests`.

### 12a. Find one representative statistic value

Start with the Previous Close value. Its `data-field` attribute is more stable than a generated CSS class.

In [160]:
previous_close_value = soup.find(
    "fin-streamer",
    attrs={"data-field": "regularMarketPreviousClose"}
)
previous_close_value

<fin-streamer active="" class="yf-11op525" data-field="regularMarketPreviousClose" data-trend="none" data-value="344.17">344.17 </fin-streamer>

### 12b. Inspect its statistic row

In the quote-statistics grid, the value and its label are local to the same containing `<li>` element.

In [161]:
previous_close_row = previous_close_value.find_parent("li") if previous_close_value else None
previous_close_row

<li class="yf-11op525"><span class="label yf-11op525" title="Previous Close"><span class="labelText yf-11op525">Previous Close</span> </span> <span class="value yf-11op525" title="344.17"><fin-streamer active="" class="yf-11op525" data-field="regularMarketPreviousClose" data-trend="none" data-value="344.17">344.17 </fin-streamer></span> </li>

### 12c. Extract one label/value pair

Use the label span in that same row instead of depending on a CSS class name.

In [162]:
previous_close_label = previous_close_row.find("span") if previous_close_row else None
previous_close_label

<span class="label yf-11op525" title="Previous Close"><span class="labelText yf-11op525">Previous Close</span> </span>

In [163]:
previous_close_record = {
    "Statistic": text_or_none(previous_close_label),
    "Value": text_or_none(previous_close_value),
}
previous_close_record

{'Statistic': 'Previous Close', 'Value': '344.17'}

### 12d. Locate the available statistic rows

Each data-backed value has a semantic `data-field`. Find those values, then move only to their containing `<li>` rows. This avoids relying on volatile generated classes.

In [164]:
quote_statistic_fields = [
    "regularMarketPreviousClose", "regularMarketOpen", "bid", "ask",
    "regularMarketDayRange", "fiftyTwoWeekRange", "regularMarketVolume",
    "averageDailyVolume3Month", "marketCap", "beta", "trailingPE",
    "epsTrailingTwelveMonths", "earningsTimestamp", "dividendRate",
    "dividendYield", "trailingAnnualDividendRate", "trailingAnnualDividendYield", "targetMeanPrice",
]

In [165]:
statistic_value_tags = soup.find_all(
    attrs={"data-field": quote_statistic_fields}
)
len(statistic_value_tags)

9

In [166]:
quote_statistic_rows = []
for value_tag in statistic_value_tags:
    row = value_tag.find_parent("li")
    if row is not None and row not in quote_statistic_rows:
        quote_statistic_rows.append(row)

len(quote_statistic_rows)

9

Morningstar Rating may appear without a `data-field`. Add its `<li>` only when the label is in the downloaded HTML. If Yahoo shows only an icon or access prompt instead of a readable rating, the final checks will skip it.

In [167]:
morningstar_label = soup.find(
    string=lambda text: text and text.strip() == "Morningstar Rating"
)
morningstar_label

'Morningstar Rating  '

In [168]:
morningstar_row = morningstar_label.find_parent("li") if morningstar_label else None
if morningstar_row is not None and morningstar_row not in quote_statistic_rows:
    quote_statistic_rows.append(morningstar_row)

### 12e. Make one small record for each available row

Skip rows whose label or value is missing. For normal quote statistics, the value comes from `data-field`; the fallback lets a readable optional Morningstar value be included.

In [169]:
quote_statistic_records = []
missing_values = {"", "-", "--", "N/A", "n/a"}

for row in quote_statistic_rows:
    label_tag = row.find(["span", "a"])
    value_tag = row.find(attrs={"data-field": True})
    label = text_or_none(label_tag)
    value = text_or_none(value_tag)

    if value is None and label:
        other_text = [text for text in row.stripped_strings if text != label]
        value = " ".join(other_text)

    if label and value and value not in missing_values:
        quote_statistic_records.append({"Statistic": label, "Value": value})

quote_statistic_records

[{'Statistic': 'Previous Close', 'Value': '344.17'},
 {'Statistic': 'Open', 'Value': '338.30'},
 {'Statistic': "Day's Range", 'Value': '338.30 - 345.79'},
 {'Statistic': '52 Week Range', 'Value': '243.42 - 355.38'},
 {'Statistic': 'Volume', 'Value': '24,285,179'},
 {'Statistic': 'Market Cap (intraday)', 'Value': '5.003T'},
 {'Statistic': 'PE Ratio (TTM)', 'Value': '38.34'},
 {'Statistic': 'EPS (TTM)', 'Value': '8.98'},
 {'Statistic': '1y Target Est', 'Value': '332.68'},
 {'Statistic': 'Morningstar Rating',
  'Value': 'Rating available with Stock Picks access'}]

### 12f. Display the label/value table

Each row of `quote_statistics` is one available Yahoo Finance quote statistic.

In [170]:
quote_statistics = pd.DataFrame(quote_statistic_records)
quote_statistics

,Statistic,Value
0,Previous Close,344.17
1,Open,338.30
2,Day's Range,338.30 - 345.79
3,52 Week Range,243.42 - 355.38
4,Volume,"24,285,179"
5,Market Cap (intraday),5.003T
6,PE Ratio (TTM),38.34
7,EPS (TTM),8.98
8,1y Target Est,332.68
9,Morningstar Rating,Rating available with Stock Picks access


## Checkpoint

You practiced **fetch → parse → inspect a tag → `find` → `find_all` → spans → tag properties → filter results → reusable logic → CSS selectors → pandas → quote-statistics grid extraction**. The next notebook applies the same sequence to Wikipedia's S&P 500 table.